# LLM-driven sessions on Colab (dataset step 7)

Runs `qwen2.5:7b` (Ollama) as the agent against the **in-process** gateway and sandbox, labels
every session from the events that actually happened, and writes samples to Google Drive.

* **Runtime:** GPU (T4). *Runtime > Change runtime type > T4 GPU* before running anything.
* **Resumable:** every session is written to Drive as soon as it finishes. After a disconnect,
  open the notebook again, run all cells, and the run continues. A run only resumes output of
  the **same `build_id`**; otherwise it stops with a clear message (use a new `OUT_DIR`).
* **Phases:** `PHASE = "all"` for a trial. For the FINAL run use two passes so the pinned commit
  contains the paraphrases: pass 1 `PHASE = "paraphrase"` (then commit `paraphrases.json`),
  pass 2 `PHASE = "sessions"` with `COMMIT` set to that new commit.
* **Everything is fake:** secrets, customers and attacks target the sandbox only.

In [ ]:
# ---- parameters ---------------------------------------------------------------------------
REPO_URL = "https://github.com/kdt523/Sentinel-mcp.git"
COMMIT = "main"            # FINAL run: pin an exact commit hash
PHASE = "all"              # "all" | "paraphrase" | "sessions"
N_BENIGN, N_ATTACK = 270, 180
AGENT_MODEL = "qwen2.5:7b"
PARAPHRASE_MODEL = "qwen2.5:3b"
SYSTEM = "plain"           # "plain" (naive agent, attacks can succeed) or "hardened"
OUT_DIR = "/content/drive/MyDrive/sentinel_llm_sessions"   # use a NEW folder for each build

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import subprocess

def sh(cmd: str, cwd: str = "/content/Sentinel-mcp"):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=cwd)
    print(r.stdout[-3000:], r.stderr[-1500:] if r.returncode else "")
    r.check_returncode()

sh(f"rm -rf Sentinel-mcp && git clone -q {REPO_URL} Sentinel-mcp", cwd="/content")
sh(f"git checkout -q {COMMIT} && git log --oneline -1")
sh("pip install -q uv")
sh('uv pip install --system -q -e ".[agents,tools]" aiosqlite pytest pytest-asyncio')
sh("python -c 'import sys; print(sys.version)'")

## Ollama with the agent model

In [ ]:
import os, subprocess, time, urllib.request
sh("apt-get install -y -qq zstd >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1; ollama --version")
env = {**os.environ, "OLLAMA_KEEP_ALIVE": "2h"}  # keep both models loaded between requests
server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT, env=env)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
    except OSError:
        time.sleep(1)
sh(f"ollama pull {AGENT_MODEL} && ollama pull {PARAPHRASE_MODEL}")
sh(f"ollama show {AGENT_MODEL} | head -20")
sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv")

## Warm up both models and check they run on the GPU
In `ollama ps` the PROCESSOR column must say **100% GPU**. If it says CPU, stop, switch the
runtime to a T4 GPU and run again from the top.

In [ ]:
import json, time, urllib.request

def ask(model: str) -> None:
    t = time.time()
    req = urllib.request.Request(
        "http://127.0.0.1:11434/api/generate",
        data=json.dumps({"model": model, "prompt": "Say hi in three words.", "stream": False}).encode(),
        headers={"Content-Type": "application/json"},
    )
    out = json.load(urllib.request.urlopen(req, timeout=900))
    print(f"{model}: {time.time() - t:.1f}s -> {out['response'][:60]!r}")

ask(PARAPHRASE_MODEL)
ask(AGENT_MODEL)
sh("ollama ps")
sh("tail -n 12 /content/ollama.log")

## Sanity check (unit tests with a scripted stand-in model; nothing here becomes data)

In [ ]:
sh("python -m pytest tests/unit/test_llm_sessions.py tests/unit/test_llm_label.py tests/unit/test_paraphrase.py tests/unit/test_llm_stats.py -q -p no:randomly 2>&1 | tail -5")

## 1. Paraphrase the benign prompts  (`PHASE` = `all` or `paraphrase`)
Rewrites that drop a path, table, address, id, number or slot word, or that only change case or
spelling, are rejected. Progress is saved as it goes and the step can be re-run.

In [ ]:
if PHASE in ("all", "paraphrase"):
    sh(f"python -m data.paraphrase --model {PARAPHRASE_MODEL} --benign {N_BENIGN} --attack {N_ATTACK}")
    sh(f"cp data/benign/paraphrases.json {OUT_DIR}/paraphrases.json")
    d = json.load(open("/content/Sentinel-mcp/data/benign/paraphrases.json"))
    print(len(d["prompts"]), "accepted,", len(d["rejected"]), "rejected")
    for k, v in list(d["rejected"].items())[:5]:
        print("  rejected", k, "->", v)
    if PHASE == "paraphrase":
        print("\nNEXT: download paraphrases.json, commit it to the repo, then run again with "
              "PHASE = 'sessions' and COMMIT = <that new commit hash>.")
else:
    print("skipped (PHASE =", PHASE + ")")

## 2. Run the sessions  (`PHASE` = `all` or `sessions`)
Safe to re-run any time: finished sessions are skipped, half-written ones are repaired, and output
from a different build is refused.

In [ ]:
if PHASE in ("all", "sessions"):
    sh(f"python -m data.llm_sessions --benign {N_BENIGN} --attack {N_ATTACK} --model {AGENT_MODEL} "
       f"--system {SYSTEM} --out {OUT_DIR}/llm_samples.jsonl --log {OUT_DIR}/llm_sessions.jsonl")
else:
    print("skipped (PHASE =", PHASE + ")")

## 3. Summary: how often did the attack actually fire?
An attack session that did not fire is labelled by what happened (all events benign) and is never
counted as a gateway catch. A family firing in under 20% of its sessions is flagged.

In [ ]:
log = f"{OUT_DIR}/llm_sessions.jsonl"
if os.path.exists(log):
    sh(f"python -m data.llm_stats --log {log} --out {OUT_DIR}/llm_stats.json")
    sh(f"git rev-parse HEAD; ollama show {AGENT_MODEL} --modelfile | grep -i '^FROM' | head -2")
else:
    print("no sessions yet")

## 4. Bring the results back
Files in `OUT_DIR` on Google Drive: `paraphrases.json`, `llm_samples.jsonl`, `llm_sessions.jsonl`,
`llm_stats.json`. Put `paraphrases.json` in `data/benign/` and commit it; the other three go to
`data/raw/` on the laptop (git-ignored). The cell below also downloads the small ones.

In [ ]:
from google.colab import files
for name in ("paraphrases.json", "llm_sessions.jsonl", "llm_stats.json"):
    if os.path.exists(f"{OUT_DIR}/{name}"):
        files.download(f"{OUT_DIR}/{name}")